# 01. Weather Feature Engineering
This notebook extracts weather features from ERA5 NetCDF files (`.nc`) for Boundary Layer Height (BLH), Relative Humidity at 850hPa (RH850), and Temperature Inversion.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path("../../")
RAW_DIR = ROOT / "data/raw"
PROC_DIR = ROOT / "data/processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

STATIONS = {
    2161292: (21.0152, 105.7999),
    2161306: (21.0500, 105.7400),
    4946811: (21.0491, 105.8831),
    4946812: (21.0031, 105.7947),
    4946813: (21.0052, 105.8418),
    6123215: (20.9933, 105.9441),
}


## 1. Process Boundary Layer Height (BLH)
We extract mean, min, max, and morning BLH values.

In [2]:
def nearest_idx(arr, target):
    return int(np.argmin(np.abs(arr - target)))

def process_one_nc_blh(nc_path):
    ds = xr.open_dataset(nc_path)
    lats = ds["latitude"].values
    lons = ds["longitude"].values
    var = [v for v in ds.data_vars if "blh" in v.lower() or "boundary" in v.lower()]
    var = var[0] if var else list(ds.data_vars)[0]
    time_coord = "valid_time" if "valid_time" in ds.coords else "time"
    da = ds[var]
    if "expver" in da.dims:
        da = da.isel(expver=0)
    if "number" in da.dims:
        da = da.isel(number=0)

    times = pd.to_datetime(da[time_coord].values)
    records = []
    for loc_id, (lat, lon) in STATIONS.items():
        il = nearest_idx(lats, lat)
        ij = nearest_idx(lons, lon)
        blh_vals = da.isel(latitude=il, longitude=ij).values.flatten().astype(float)
        
        df_hr = pd.DataFrame({"time": times, "blh": blh_vals})
        df_hr["date"] = df_hr["time"].dt.normalize()
        df_hr["is_morning"] = df_hr["time"].dt.hour.isin([23, 0, 1, 2])
        df_hr["date_vn"] = pd.to_datetime(df_hr.apply(
            lambda r: r["time"].date() + pd.Timedelta(days=1)
            if r["time"].hour == 23 else r["time"].date(), axis=1
        ))

        daily = (
            df_hr.groupby("date")["blh"]
            .agg(blh_mean="mean", blh_min="min", blh_max="max")
            .reset_index()
        )
        morn = (
            df_hr[df_hr["is_morning"]]
            .groupby("date_vn")["blh"].mean()
            .rename("blh_morning").reset_index()
            .rename(columns={"date_vn": "date"})
        )
        daily = daily.merge(morn, on="date", how="left")
        daily.insert(0, "location_id", loc_id)
        records.append(daily)

    ds.close()
    return pd.concat(records, ignore_index=True)

nc_parts = [RAW_DIR / 'era5' / f'era5_blh_{yr}.nc' for yr in ['2024', '2025', '2026']]
existing = [p for p in nc_parts if p.exists()]
if not existing:
    print("No BLH NetCDF files found.")
else:
    parts_df = [process_one_nc_blh(p) for p in existing]
    blh_df = (
        pd.concat(parts_df, ignore_index=True)
        .drop_duplicates(subset=["location_id", "date"])
        .sort_values(["location_id", "date"])
        .reset_index(drop=True)
    )
    blh_df.to_csv(PROC_DIR / "04_era5_blh_daily.csv", index=False)
    blh_df.to_csv(PROC_DIR / "era5_blh_daily.csv", index=False)
    print(f"Saved BLH daily. Shape: {blh_df.shape}")


Saved BLH daily. Shape: (5190, 6)


## 2. Process Relative Humidity at 850hPa (RH850)
We extract mean, max, and morning values of RH850.

In [3]:
def process_rh850_file(nc_path):
    ds = xr.open_dataset(nc_path)
    var = "r"
    if var not in ds.data_vars:
        var = list(ds.data_vars)[0]
    da = ds[var]
    if "expver" in da.dims: da = da.isel(expver=0)
    if "number" in da.dims: da = da.isel(number=0)
    if "pressure_level" in da.dims: da = da.isel(pressure_level=0)
    if "level" in da.dims: da = da.isel(level=0)

    tc = "valid_time" if "valid_time" in ds.coords else "time"
    lats = da.latitude.values
    lons = da.longitude.values
    times = pd.to_datetime(da[tc].values)

    records = []
    for it, t in enumerate(times):
        for ilat, lat in enumerate(lats):
            for ilon, lon in enumerate(lons):
                val = float(da.values[it, ilat, ilon])
                records.append({"time": t, "latitude": lat, "longitude": lon, "rh850": val})
    return pd.DataFrame(records)

def nearest_grid(df_hourly, lat, lon):
    lats = df_hourly["latitude"].unique()
    lons = df_hourly["longitude"].unique()
    best_lat = lats[np.argmin(np.abs(lats - lat))]
    best_lon = lons[np.argmin(np.abs(lons - lon))]
    return df_hourly[(df_hourly["latitude"] == best_lat) &
                     (df_hourly["longitude"] == best_lon)].copy()

files = sorted(list((RAW_DIR / "era5_rh850").glob("era5_rh850_????_??.nc")))
if not files:
    print("No RH850 NetCDF files found.")
else:
    print(f"Processing {len(files)} files ...")
    all_station_dfs = {loc_id: [] for loc_id in STATIONS}
    for nc_path in files:
        df_hr = process_rh850_file(nc_path)
        for loc_id, (lat, lon) in STATIONS.items():
            df_st = nearest_grid(df_hr, lat, lon)
            df_st = df_st[["time", "rh850"]].copy()
            df_st.insert(0, "location_id", loc_id)
            all_station_dfs[loc_id].append(df_st)

    daily_records = []
    for loc_id, chunks in all_station_dfs.items():
        df_st = pd.concat(chunks, ignore_index=True).drop_duplicates("time")
        df_st["time"] = pd.to_datetime(df_st["time"])
        df_st["date"] = df_st["time"].dt.normalize()
        df_st["is_morning"] = df_st["time"].dt.hour.isin([6, 7, 8, 9])

        daily = (df_st.groupby("date")["rh850"]
                 .agg(rh850_mean="mean", rh850_max="max")
                 .reset_index())
        morn = (df_st[df_st["is_morning"]]
                 .groupby("date")["rh850"].mean()
                 .rename("rh850_morning").reset_index())
        daily = daily.merge(morn, on="date", how="left")
        daily.insert(0, "location_id", loc_id)
        daily_records.append(daily)

    rh_df = (pd.concat(daily_records, ignore_index=True)
             .sort_values(["location_id", "date"])
             .reset_index(drop=True))
    rh_df["month"] = rh_df["date"].dt.month
    rh_df.to_csv(PROC_DIR / "05_era5_rh850_daily.csv", index=False)
    rh_df.to_csv(PROC_DIR / "era5_rh850_daily.csv", index=False)
    print(f"Saved RH850 daily. Shape: {rh_df.shape}")


Processing 29 files ...
Saved RH850 daily. Shape: (5196, 6)


## 3. Process Temperature Inversion
We extract temperature values at 850hPa, 925hPa, and 1000hPa pressure levels, and compute inversion strengths.

In [4]:
def process_nc_tinv(nc_path):
    ds = xr.open_dataset(nc_path)
    lats = ds["latitude"].values
    lons = ds["longitude"].values
    tc = "valid_time" if "valid_time" in ds.coords else "time"
    var = [v for v in ds.data_vars if "t" in v.lower() or "temp" in v.lower()]
    var = var[0] if var else list(ds.data_vars)[0]
    lev_coord = None
    for c in ["pressure_level", "level", "plev"]:
        if c in ds.coords or c in ds.dims:
            lev_coord = c; break
    da = ds[var]
    for dim in ["expver", "number"]:
        if dim in da.dims:
            da = da.isel({dim: 0})

    times = pd.to_datetime(da[tc].values)
    levels = da[lev_coord].values.astype(float) if lev_coord else None

    def get_level(hpa):
        return nearest_idx(levels, hpa) if levels is not None else 0

    i850 = get_level(850)
    i925 = get_level(925)
    i1000 = get_level(1000)

    records = []
    for loc_id, (lat, lon) in STATIONS.items():
        il = nearest_idx(lats, lat)
        ij = nearest_idx(lons, lon)

        def get_t(ilev):
            if lev_coord:
                vals = da.isel({lev_coord: ilev, "latitude": il, "longitude": ij})
            else:
                vals = da.isel(latitude=il, longitude=ij)
            return vals.values.flatten().astype(float) - 273.15

        t850 = get_t(i850)
        t925 = get_t(i925)
        t1000 = get_t(i1000)

        df_hr = pd.DataFrame({
            "time": times,
            "t850": t850,
            "t925": t925,
            "t1000": t1000,
        })
        df_hr["date"] = df_hr["time"].dt.normalize()
        df_hr["inv_850_1000"] = df_hr["t850"] - df_hr["t1000"]
        df_hr["inv_925_1000"] = df_hr["t925"] - df_hr["t1000"]
        df_hr["inv_850_925"] = df_hr["t850"] - df_hr["t925"]

        agg_cols = ["t850", "t925", "t1000", "inv_850_1000", "inv_925_1000", "inv_850_925"]
        daily = df_hr.groupby("date")[agg_cols].agg(["mean", "max"]).reset_index()
        daily.columns = ["date"] + [f'{c}_{s}' for c, s in daily.columns[1:]]

        df_hr["is_morning"] = df_hr["time"].dt.hour.isin([23, 0, 1, 2])
        morn = (df_hr[df_hr["is_morning"]]
                .groupby("date")["inv_850_1000"].mean()
                .rename("inv_850_1000_morning").reset_index())
        daily = daily.merge(morn, on="date", how="left")
        daily.insert(0, "location_id", loc_id)
        records.append(daily)

    ds.close()
    return pd.concat(records, ignore_index=True)

files = sorted(list((RAW_DIR / "era5_tinv").glob("era5_tinv_*.nc")))
if not files:
    print("No temperature inversion NetCDF files found.")
else:
    parts = [process_nc_tinv(p) for p in files]
    tinv_df = (pd.concat(parts, ignore_index=True)
               .drop_duplicates(subset=["location_id", "date"])
               .sort_values(["location_id", "date"])
               .reset_index(drop=True))
    
    keep_cols = ["location_id", "date", "t850_mean", "t925_mean", "t1000_mean", "inv_850_1000_mean", "inv_850_1000_max", "inv_925_1000_mean", "inv_850_1000_morning"]
    tinv_df = tinv_df[keep_cols]
    
    tinv_df.to_csv(PROC_DIR / "06_era5_t_inversion_daily.csv", index=False)
    tinv_df.to_csv(PROC_DIR / "era5_t_inversion_daily.csv", index=False)
    print(f"Saved Temp Inversion daily. Shape: {tinv_df.shape}")


Saved Temp Inversion daily. Shape: (5196, 9)
